# 单元格 1：安装依赖并导入库

In [ ]:
!pip install jieba scikit-learn -i https://mirror.baidu.com/pypi/simple

import re
import jieba
import numpy as np
import pandas as pd
import paddle
import paddle.nn as nn
from paddle.io import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

# 单元格 2：加载原始数据

In [ ]:
# 请确认数据路径，如不同请修改
base_path = 'data/sentiment_data/ChnSentiCorp/ChnSentiCorp/'

train_df = pd.read_csv(base_path + 'train.tsv', sep='\t')
dev_df   = pd.read_csv(base_path + 'dev.tsv', sep='\t')

print("训练集列名：", train_df.columns.tolist())
print("开发集列名：", dev_df.columns.tolist())

train_texts_raw = train_df['text_a'].astype(str).tolist()
train_labels_raw = train_df['label'].astype(int).tolist()

dev_texts_raw = dev_df['text_a'].astype(str).tolist()
dev_labels_raw = dev_df['label'].astype(int).tolist()

print(f"原始训练集样本数：{len(train_texts_raw)}")
print(f"开发集样本数：{len(dev_texts_raw)}")

# 单元格 3：文本预处理函数与词表构建

In [ ]:
def clean_text(text):
    text = re.sub(r'[^\u4e00-\u9fa5a-zA-Z0-9]', '', str(text))
    return text.strip()

def tokenize(text):
    return jieba.lcut(clean_text(text))

def build_vocab(texts, max_vocab=5000):
    all_words = []
    for text in texts:
        all_words.extend(tokenize(text))
    word_counts = Counter(all_words)
    vocab = {'<pad>': 0, '<unk>': 1}
    for word, _ in word_counts.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_texts_raw, max_vocab=5000)
print(f"词表大小：{len(vocab)}")

def text_to_ids(text, vocab, max_len=100):
    words = tokenize(text)
    ids = [vocab.get(w, vocab['<unk>']) for w in words[:max_len]]
    ids += [vocab['<pad>']] * (max_len - len(ids))
    return np.array(ids, dtype='int64')

# 单元格 4：划分训练/验证集并转换为ID序列

In [ ]:
train_ids = np.array([text_to_ids(t, vocab, max_len) for t in train_texts_raw])
val_ids   = np.array([text_to_ids(t, vocab, max_len) for t in dev_texts_raw])
train_labels = np.array(train_labels_raw, dtype='int64')
val_labels   = np.array(dev_labels_raw, dtype='int64')

# 单元格 5：构建 DataLoader

In [ ]:
batch_size = 64

train_dataset = TensorDataset([paddle.to_tensor(train_ids), paddle.to_tensor(train_labels)])
val_dataset   = TensorDataset([paddle.to_tensor(val_ids),   paddle.to_tensor(val_labels)])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# 单元格 6：定义模型（必须与保存时结构一致）

In [ ]:
class SentimentLSTM(nn.Layer):
    def __init__(self, vocab_size, embed_dim=128, hidden_size=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_size, 
                            num_layers=1, 
                            direction='bidirectional')
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x, (h, c) = self.lstm(x)
        x = paddle.mean(x, axis=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

model = SentimentLSTM(vocab_size=len(vocab))

# 单元格 7：加载已保存的模型参数

In [ ]:
# 假设模型文件在当前目录下
model.load_dict(paddle.load('sentiment_lstm.pdparams'))
print("✅ 模型参数加载成功！")

# 单元格 8：评估已加载模型的当前性能

In [ ]:
model.eval()
all_preds = []
all_labels = []
with paddle.no_grad():
    for batch_x, batch_y in val_loader:
        logits = model(batch_x)
        preds = paddle.argmax(logits, axis=1)
        all_preds.extend(preds.numpy().tolist())
        all_labels.extend(batch_y.numpy().tolist())

acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds)
print(f"当前验证集准确率：{acc:.4f}，F1值：{f1:.4f}")

# 开始：进一步调试与性能提升

### 方向一：调整超参数重新训练

In [ ]:
model = SentimentLSTM(vocab_size=len(vocab))  # 重新初始化参数
criterion = nn.CrossEntropyLoss()
optimizer = paddle.optimizer.Adam(learning_rate=0.0005, parameters=model.parameters())  # 降低学习率

epochs = 25
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    
    # 验证
    model.eval()
    correct, total = 0, 0
    with paddle.no_grad():
        for batch_x, batch_y in val_loader:
            pred = model(batch_x)
            pred_label = paddle.argmax(pred, axis=1)
            correct += (pred_label == batch_y).sum().item()
            total += batch_y.shape[0]
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")

### 方向二：改进模型结构

In [ ]:
class Attention(nn.Layer):
    def __init__(self, hidden_size):
        super().__init__()
        self.att_weight = self.create_parameter([hidden_size*2, 1])
    def forward(self, lstm_out):
        # lstm_out: [batch, seq_len, hidden*2]
        scores = paddle.matmul(lstm_out, self.att_weight).squeeze(-1)
        att_weights = paddle.nn.functional.softmax(scores, axis=1)
        context = paddle.sum(lstm_out * att_weights.unsqueeze(-1), axis=1)
        return context

In [ ]:
model = SentimentLSTM(vocab_size=len(vocab))  # 重新初始化参数
criterion = nn.CrossEntropyLoss()
optimizer = paddle.optimizer.Adam(learning_rate=0.0004, parameters=model.parameters())  # 降低学习率

epochs = 25
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        pred = model(batch_x)
        loss = criterion(pred, batch_y)
        loss.backward()
        optimizer.step()
        optimizer.clear_grad()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    
    # 验证
    model.eval()
    correct, total = 0, 0
    with paddle.no_grad():
        for batch_x, batch_y in val_loader:
            pred = model(batch_x)
            pred_label = paddle.argmax(pred, axis=1)
            correct += (pred_label == batch_y).sum().item()
            total += batch_y.shape[0]
    acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Val Acc: {acc:.4f}")